# Agricultural Crop Yield Prediction

This notebook demonstrates the Big Data Analytics project for predicting crop yield based on climate and soil data.

## 1. Import Required Libraries

Import necessary Python libraries for data processing, Spark, and analytics.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.regression import LinearRegression
from pymongo import MongoClient
import requests
from bs4 import BeautifulSoup

# Set up Spark session
spark = SparkSession.builder \
    .appName("Crop Yield Prediction") \
    .getOrCreate()

print("Libraries imported successfully!")

## 2. Collect Government Datasets on Crop Yield

Download the Kaggle Crop Yield Prediction Dataset.

In [ ]:
# Download dataset from Kaggle (assuming kaggle.json is set up)
# For notebook, we'll assume the data is already downloaded to ../data/
# If not, run the download script

data_path = '../data/yield.csv'  # Adjust based on actual file name
df = pd.read_csv(data_path)
print("Dataset loaded successfully!")
print(df.head())

## 3. Store Data in HDFS

Upload the dataset to HDFS for distributed processing.

In [ ]:
# Convert to Spark DataFrame and save to HDFS
spark_df = spark.createDataFrame(df)
spark_df.write.csv("hdfs://localhost:9000/crop_yield/raw_data", header=True, mode="overwrite")
print("Data stored in HDFS!")

## 4. Use MapReduce to Clean and Preprocess Data

Clean the data using Spark (MapReduce equivalent).

In [ ]:
# Load data from HDFS
raw_data = spark.read.csv("hdfs://localhost:9000/crop_yield/raw_data", header=True, inferSchema=True)

# Clean data: drop nulls, etc.
cleaned_data = raw_data.dropna()

# Encode categorical variables
indexer = StringIndexer(inputCol="soil_type", outputCol="soil_type_encoded")
cleaned_data = indexer.fit(cleaned_data).transform(cleaned_data)

# Save cleaned data
cleaned_data.write.csv("hdfs://localhost:9000/crop_yield/cleaned_data", header=True, mode="overwrite")
print("Data cleaned and saved!")

## 5. Use Hive for Trend Queries

Analyze trends using Hive-like queries in Spark SQL.

In [ ]:
# Create temp view for SQL queries
cleaned_data.createOrReplaceTempView("crop_yield")

# Query 1: Average yield by crop
avg_yield = spark.sql("SELECT crop, AVG(yield) as avg_yield FROM crop_yield GROUP BY crop")
avg_yield.show()

# Query 2: Yield over years
yield_trend = spark.sql("SELECT year, AVG(yield) as avg_yield FROM crop_yield GROUP BY year ORDER BY year")
yield_trend.show()

## 6. Store Refined Data in MongoDB

Export cleaned data to MongoDB.

In [ ]:
# Convert to Pandas and insert to MongoDB
client = MongoClient('localhost', 27017)
db = client['crop_yield_db']
collection = db['yield_data']

pandas_df = cleaned_data.toPandas()
data_dict = pandas_df.to_dict('records')
collection.insert_many(data_dict)
print("Data stored in MongoDB!")

## 7. Apply Spark MLlib for Yield Prediction

Train a regression model to predict yield.

In [ ]:
# Prepare features
feature_cols = ['temperature', 'rainfall', 'soil_type_encoded']
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
data = assembler.transform(cleaned_data)

# Split data
train_data, test_data = data.randomSplit([0.8, 0.2], seed=42)

# Train model
lr = LinearRegression(featuresCol='features', labelCol='yield')
lr_model = lr.fit(train_data)

# Predict
predictions = lr_model.transform(test_data)
predictions.select("yield", "prediction").show()

# Evaluate
from pyspark.ml.evaluation import RegressionEvaluator
evaluator = RegressionEvaluator(labelCol="yield", predictionCol="prediction", metricName="rmse")
rmse = evaluator.evaluate(predictions)
print(f"RMSE: {rmse}")

## 8. Perform Web Analytics on Farming Trends

Scrape farming trend data from the web.

In [ ]:
# Scrape farming trends
url = "https://www.farmbrite.com/blog/farming-trends"  # Example URL
response = requests.get(url)
soup = BeautifulSoup(response.text, 'html.parser')

# Extract trends (adjust selectors)
trends = []
for item in soup.find_all('h3'):
    trends.append(item.text.strip())

print("Farming Trends:")
for trend in trends[:5]:  # Show first 5
    print(trend)